# RealWorld HAR — Classical ML Model Comparison (LOSO Cross-Validation)

This notebook trains and compares several classical machine learning
models on the extracted feature table (`realworld_features.pkl`) using
Leave-One-Subject-Out (LOSO) cross-validation.

Models compared:
- Random Forest
- Gradient Boosting
- Support Vector Machine (RBF kernel)
- K-Nearest Neighbors
- Logistic Regression

Pipeline:
1. Load features, prepare X / y / groups
2. Define a model registry (name → estimator)
3. Run one shared LOSO loop per model
4. Aggregate metrics into a comparison table
5. Visualize results (bar chart, per-fold F1, confusion matrices)

In [1]:
import time
import numpy as np
import pandas as pd

from pathlib import Path

from sklearn.model_selection import LeaveOneGroupOut
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 50)
plt.rcParams["figure.figsize"] = (10, 6)

In [2]:
DATA_DIR = Path("/lustre09/project/6081099/reem2005/DATASET/PROCESSED")
FEATURES_INPUT = DATA_DIR / "realworld_features.pkl"

features_df = pd.read_pickle(FEATURES_INPUT)

print("Shape:", features_df.shape)
print("Any missing values:", features_df.isna().sum().sum())
features_df.head()

Shape: (435295, 169)
Any missing values: 0


,body_x_mean,body_x_std,body_x_var,body_x_min,body_x_max,body_x_median,body_x_range,body_x_iqr,body_x_mad,body_x_rms,body_x_energy,body_x_skew,body_x_kurtosis,body_x_zero_cross,body_x_spectral_energy,body_x_dominant_freq,body_x_spectral_entropy,body_y_mean,body_y_std,body_y_var,body_y_min,body_y_max,body_y_median,body_y_range,body_y_iqr,...,mag_z_mad,mag_z_rms,mag_z_energy,mag_z_skew,mag_z_kurtosis,mag_z_zero_cross,mag_z_spectral_energy,mag_z_dominant_freq,mag_z_spectral_entropy,body_sma,gyr_sma,mag_sma,body_corr_xy,body_corr_xz,body_corr_yz,gyr_corr_xy,gyr_corr_xz,gyr_corr_yz,mag_corr_xy,mag_corr_xz,mag_corr_yz,participant,activity,position,segment_id
0,-0.013312,0.034942,0.001221,-0.099193,0.067895,-0.014541,0.167088,0.037801,0.026692,0.037392,0.139817,-0.041324,-0.101899,26,0.002442,0.5,4.289833,-0.001022,0.052340,0.002740,-0.196473,0.124799,-0.002177,0.321272,0.051679,...,0.000000,8.541000,7294.868100,0.000000,0.000000,0,0.000000,0.0,-0.000000,0.130143,0.065356,66.20488,-0.056975,0.043867,0.152290,-0.120569,0.453541,-0.613501,-1.145233e-14,0.000000e+00,0.000000e+00,proband1,climbingdown,chest,1
1,0.001922,0.043612,0.001902,-0.099193,0.118463,0.002581,0.217656,0.049927,0.034279,0.043654,0.190567,0.130511,-0.001179,15,0.003804,0.5,4.059204,0.001261,0.054277,0.002946,-0.196473,0.124799,-0.000502,0.321272,0.057962,...,0.000000,8.541000,7294.868100,0.000000,0.000000,0,0.000000,0.0,-0.000000,0.129010,0.065506,66.11609,-0.064799,-0.099952,0.202797,-0.119754,0.477623,-0.592597,-3.000504e-14,0.000000e+00,0.000000e+00,proband1,climbingdown,chest,1
2,0.006834,0.042727,0.001826,-0.077613,0.118463,0.005763,0.196076,0.064080,0.034446,0.043270,0.187232,0.303122,-0.336565,17,0.003651,1.0,4.272679,-0.002082,0.048478,0.002350,-0.158838,0.125513,-0.003773,0.284351,0.056050,...,0.078285,8.587750,7374.944445,1.960392,1.843137,0,0.024033,0.5,2.827348,0.126286,0.065775,66.21232,0.102075,-0.057787,0.081739,0.065038,0.371056,-0.463318,2.450713e-14,2.903576e-01,2.349659e-14,proband1,climbingdown,chest,1
3,-0.010086,0.038397,0.001474,-0.107254,0.106087,-0.016177,0.213341,0.046962,0.031332,0.039700,0.157606,0.356674,0.135641,28,0.002949,1.0,4.165225,0.005655,0.047445,0.002251,-0.146539,0.125513,0.006722,0.272052,0.044856,...,0.139685,8.741776,7641.865595,-0.628971,-1.604396,0,0.042883,0.5,1.707531,0.124160,0.068591,66.30111,0.187398,0.145190,0.011396,0.099018,0.405679,-0.434695,5.019650e-14,9.177195e-01,1.152457e-14,proband1,climbingdown,chest,1
4,-0.008177,0.035113,0.001233,-0.107254,0.088120,-0.007674,0.195375,0.043442,0.027607,0.036053,0.129979,0.037294,0.357301,29,0.002466,9.5,4.820722,0.004689,0.058465,0.003418,-0.216583,0.122389,0.008268,0.338971,0.056371,...,0.136435,8.745826,7648.946688,-0.675521,-1.543672,0,0.041476,0.5,1.724136,0.134652,0.068857,66.18664,0.025172,0.156597,0.219557,0.120205,0.456632,-0.304178,1.000000e+00,1.434992e-14,1.406210e-14,proband1,climbingdown,chest,1


In [3]:
DROP_COLS = ["participant", "activity", "position", "segment_id"]

X = features_df.drop(columns=DROP_COLS)
y = features_df["activity"]
groups = features_df["participant"]

print("X shape:", X.shape)
print("Number of unique participants (LOSO folds):", groups.nunique())
print("\nClass distribution:")
print(y.value_counts())

X shape: (435295, 165)
Number of unique participants (LOSO folds): 15

Class distribution:
activity
running         68882
lying           62692
sitting         62525
walking         62430
standing        61542
climbingup      59670
climbingdown    48295
jumping          9259
Name: count, dtype: int64


## Model Registry

Each model is wrapped in a `Pipeline` with a `StandardScaler` — this is
required for distance/margin-based models (SVM, KNN, Logistic Regression)
and harmless for tree-based models (RF, Gradient Boosting), so using the
same pipeline shape for every model keeps the training loop identical.

Scaling is fit **inside** each LOSO fold (via the pipeline), on training
data only — this avoids the data leakage that happened in the earlier
version of this project (global `StandardScaler.fit` before the fold split).

In [4]:
MODEL_REGISTRY = {
    "random_forest": RandomForestClassifier(
        n_estimators=300, max_depth=20, max_features="sqrt",
        class_weight="balanced", random_state=42, n_jobs=-1,
    ),
    "gradient_boosting": GradientBoostingClassifier(
        n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42,
    ),
    "svm_rbf": SVC(
        kernel="rbf", C=10, gamma="scale", class_weight="balanced", random_state=42,
    ),
    "knn": KNeighborsClassifier(
        n_neighbors=15, weights="distance", n_jobs=-1,
    ),
    "logistic_regression": LogisticRegression(
        max_iter=2000, class_weight="balanced", n_jobs=-1, random_state=42,
    ),
}

print("Models to compare:", list(MODEL_REGISTRY.keys()))

Models to compare: ['random_forest', 'gradient_boosting', 'svm_rbf', 'knn', 'logistic_regression']


## Shared LOSO Training Function

One function runs LOSO cross-validation for any given estimator, so the
same evaluation logic is applied identically to every model in the
registry — this is what makes the comparison fair.

In [5]:
def run_loso_experiment(model_name: str, estimator, X, y, groups, verbose: bool = True) -> dict:
    """Runs Leave-One-Subject-Out cross-validation for one estimator.
    Scaling is fit on training data only, inside each fold, via Pipeline."""
    logo = LeaveOneGroupOut()

    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", estimator),
    ])

    fold_accuracies, fold_precisions, fold_recalls, fold_f1_scores = [], [], [], []
    all_true, all_pred = [], []

    t0 = time.time()

    for fold, (train_idx, test_idx) in enumerate(logo.split(X, y, groups), 1):
        test_participant = groups.iloc[test_idx].iloc[0]

        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_test)

        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, average="macro", zero_division=0)
        rec = recall_score(y_test, y_pred, average="macro", zero_division=0)
        f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)

        fold_accuracies.append(acc)
        fold_precisions.append(prec)
        fold_recalls.append(rec)
        fold_f1_scores.append(f1)

        all_true.extend(y_test)
        all_pred.extend(y_pred)

        if verbose:
            print(f"  [{model_name}] Fold {fold:2d} | test={test_participant:10s} | "
                  f"Acc={acc:.4f} | F1={f1:.4f}")

    elapsed = time.time() - t0

    return {
        "model_name": model_name,
        "fold_accuracies": fold_accuracies,
        "fold_f1_scores": fold_f1_scores,
        "mean_accuracy": np.mean(fold_accuracies),
        "mean_precision": np.mean(fold_precisions),
        "mean_recall": np.mean(fold_recalls),
        "mean_f1": np.mean(fold_f1_scores),
        "std_f1": np.std(fold_f1_scores),
        "y_true": all_true,
        "y_pred": all_pred,
        "elapsed_seconds": elapsed,
    }

Main Training Loop (all models)

In [ ]:
all_results = {}

for model_name, estimator in MODEL_REGISTRY.items():
    print(f"\n{'=' * 60}")
    print(f"Training: {model_name}")
    print(f"{'=' * 60}")

    result = run_loso_experiment(model_name, estimator, X, y, groups, verbose=False)
    all_results[model_name] = result

    print(f"Mean Accuracy: {result['mean_accuracy']:.4f}")
    print(f"Mean Macro F1: {result['mean_f1']:.4f} ± {result['std_f1']:.4f}")
    print(f"Time: {result['elapsed_seconds']/60:.1f} min")


Training: random_forest


## Results Comparison Table

In [ ]:
summary_df = pd.DataFrame([
    {
        "model": name,
        "mean_accuracy": res["mean_accuracy"],
        "mean_f1_macro": res["mean_f1"],
        "std_f1_macro": res["std_f1"],
        "time_minutes": res["elapsed_seconds"] / 60,
    }
    for name, res in all_results.items()
]).sort_values("mean_f1_macro", ascending=False).reset_index(drop=True)

summary_df

In [ ]:
plt.figure(figsize=(10, 6))
plt.bar(
    summary_df["model"], summary_df["mean_f1_macro"],
    yerr=summary_df["std_f1_macro"], capsize=5, color="#55A868"
)
plt.ylabel("Macro F1 (mean across LOSO folds)")
plt.title("Classical ML Model Comparison — LOSO Cross-Validation")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
fold_data = []
for name, res in all_results.items():
    for f1 in res["fold_f1_scores"]:
        fold_data.append({"model": name, "fold_f1": f1})

fold_df = pd.DataFrame(fold_data)

plt.figure(figsize=(10, 6))
sns.boxplot(data=fold_df, x="model", y="fold_f1")
plt.ylabel("Macro F1 per fold")
plt.title("Distribution of Per-Fold F1 Scores by Model")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


## Detailed Report for the Best Model

Best Model Classification Report + Confusion Matrix

In [ ]:
best_model_name = summary_df.iloc[0]["model"]
best_result = all_results[best_model_name]

print(f"Best model: {best_model_name}")
print("=" * 60)
print(classification_report(best_result["y_true"], best_result["y_pred"], digits=4))

labels_sorted = sorted(y.unique())
cm = confusion_matrix(best_result["y_true"], best_result["y_pred"], labels=labels_sorted)
cm_normalized = cm.astype(float) / cm.sum(axis=1, keepdims=True)

plt.figure(figsize=(9, 8))
sns.heatmap(
    cm_normalized, annot=True, fmt=".2f", cmap="Blues",
    xticklabels=labels_sorted, yticklabels=labels_sorted
)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title(f"{best_model_name} — Normalized Confusion Matrix (LOSO)")
plt.tight_layout()
plt.show()

In [ ]:
RESULTS_OUTPUT = DATA_DIR / "classical_ml_loso_results.pkl"
pd.to_pickle(all_results, RESULTS_OUTPUT)
print("Saved all model results to:", RESULTS_OUTPUT)

## Summary

- Compared 5 classical ML models (Random Forest, Gradient Boosting, SVM,
  KNN, Logistic Regression) using LOSO cross-validation
- Scaling fit per-fold on training data only — no leakage
- Best model: see `summary_df` above, sorted by mean macro F1

**Next step:** compare these classical ML results against the deep
learning models (TinyHAR, DeepConvLSTM, etc.) trained on the raw-signal
windowing pipeline.